# 06 – IV Smile

Build and plot a per-expiry implied volatility smile from the sample chain.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import pandas as pd
import matplotlib.pyplot as plt
from pricer.vol_smile import build_smile, plot_smile
from pricer.svi import fit_svi, svi_iv
import numpy as np


In [ ]:
chain = pd.read_csv("../data/sample_chain.csv")
print(chain.head())
print(f"\nUnderlyings: {chain['underlying'].unique()}")
print(f"Expiries:    {chain['expiry'].unique()}")


## AAPL June 2025 smile

In [ ]:
S_aapl = 180.0
smile = build_smile(chain, expiry="2025-06-20", S=S_aapl, r=0.05, q=0.01)
print(smile.to_string(index=False))


In [ ]:
plot_smile(smile, expiry="2025-06-20", S=S_aapl)


## SVI fit to AAPL Jun 2025 smile

Fit a raw SVI parametrisation to the per-strike IV.

In [ ]:
F = S_aapl * np.exp((0.05 - 0.01) * 0.25)  # forward price
smile_nonan = smile.dropna(subset=["iv"])

k = np.log(smile_nonan["K"].values / F)
T = 0.25
w = (smile_nonan["iv"].values ** 2) * T  # total implied variance

params, rmse = fit_svi(k, w)
print(f"SVI params [a, b, rho, m, sigma]: {params.round(4)}")
print(f"RMSE (total var): {rmse:.6f}")

k_grid = np.linspace(k.min(), k.max(), 100)
iv_svi = svi_iv(k_grid, params, T)

K_grid = F * np.exp(k_grid)
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(smile_nonan["K"], smile_nonan["iv"] * 100, label="Market IV", zorder=5)
ax.plot(K_grid, iv_svi * 100, label="SVI fit")
ax.axvline(S_aapl, linestyle="--", color="grey", alpha=0.6, label="Spot")
ax.set_xlabel("Strike")
ax.set_ylabel("Implied Volatility (%)")
ax.set_title("AAPL Jun 2025 – SVI Fit")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## MSFT Sep 2025 smile

In [ ]:
S_msft = 420.0
smile_msft = build_smile(chain, expiry="2025-09-19", S=S_msft, r=0.05, q=0.008)
plot_smile(smile_msft, expiry="2025-09-19", S=S_msft)
